# 중력모델 결과 저장

이 노트북은 현재 `src/gravity(경훈)`의 중력모델을 실행하고, 1137개 행정동 전체 OD 예측 결과와 테스트 지역 포함 OD 결과를 한글 컬럼 CSV로 저장한다.

In [ ]:
import os
from pathlib import Path
import pandas as pd

os.environ['KMP_DUPLICATE_OK'] = 'True'
작업폴더 = Path.cwd()
print('현재 작업폴더:', 작업폴더)

## 1. 모델 실행 및 CSV 저장

In [ ]:
from gravity_results import 저장하기

저장하기()

## 2. 저장된 파일 확인

In [ ]:
출력폴더 = Path('outputs')

파일목록 = [
    '전체_1137개_행정동_OD_중력모델_예측.csv',
    '테스트지역포함_OD_중력모델_예측.csv',
    '전체_행정동_총량_중력모델_예측.csv',
    '테스트지역_총량_중력모델_예측.csv',
]

for 파일명 in 파일목록:
    경로 = 출력폴더 / 파일명
    df = pd.read_csv(경로, nrows=5)
    print('\n', 파일명)
    print('컬럼:', list(df.columns))
    display(df)

## 3. 테스트 지역 총량 비교

In [ ]:
테스트총량 = pd.read_csv(출력폴더 / '테스트지역_총량_중력모델_예측.csv')

볼컬럼 = [
    '행정동코드', '행정동', '테스트지역명',
    '실제외부유출', '예측외부유출_LGBM', '외부유출오차_예측-실제',
    '실제외부유입', '예측외부유입_LGBM', '외부유입오차_예측-실제',
    '실제내부통행', '예측내부통행_LGBM', '내부통행오차_예측-실제',
]

테스트총량[볼컬럼]

## 4. 테스트 지역 포함 OD 확인

In [ ]:
테스트OD = pd.read_csv(출력폴더 / '테스트지역포함_OD_중력모델_예측.csv')

테스트OD.sort_values('절대오차', ascending=False).head(30)

## 5. 성능 분석 그림 저장

아래 셀은 테스트 지역 포함 OD 결과로 구간별 RMSE/CPC와 실제-예측 산점도를 그리고, 테스트 지역 총량 결과로 내부통행 산점도를 그린다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

def cpc_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    분자 = 2 * np.minimum(y_true, y_pred).sum()
    분모 = y_true.sum() + y_pred.sum()
    return 분자 / 분모 if 분모 > 0 else 0

테스트OD = pd.read_csv(출력폴더 / '테스트지역포함_OD_중력모델_예측.csv')
테스트총량 = pd.read_csv(출력폴더 / '테스트지역_총량_중력모델_예측.csv')

구간경계 = [-1, 0, 10, 50, 200, 1000, np.inf]
구간라벨 = ['0', '1~10', '11~50', '51~200', '201~1000', '1000+']
테스트OD['실제통행량구간'] = pd.cut(테스트OD['실제통행량'], bins=구간경계, labels=구간라벨)

구간결과 = []
for 구간 in 구간라벨:
    부분 = 테스트OD[테스트OD['실제통행량구간'] == 구간]
    if len(부분) == 0:
        continue
    실제 = 부분['실제통행량'].to_numpy(float)
    예측 = 부분['예측통행량_중력모델'].to_numpy(float)
    구간결과.append({
        '실제통행량구간': 구간,
        '개수': len(부분),
        '실제평균': 실제.mean(),
        '중력모델_RMSE': np.sqrt(np.mean((실제 - 예측) ** 2)),
        '중력모델_CPC': cpc_score(실제, 예측),
    })

구간결과 = pd.DataFrame(구간결과)
display(구간결과)

plt.figure(figsize=(16, 10))

plt.subplot(2, 2, 1)
sns.barplot(data=구간결과, x='실제통행량구간', y='중력모델_RMSE', color='green', alpha=0.8)
plt.title('실제 통행량 구간별 RMSE (테스트 지역)')
plt.xlabel('실제 통행량 구간')
plt.ylabel('중력모델 RMSE')
plt.xticks(rotation=45)

plt.subplot(2, 2, 2)
sns.barplot(data=구간결과, x='실제통행량구간', y='중력모델_CPC', color='green', alpha=0.8)
plt.title('실제 통행량 구간별 CPC (테스트 지역)')
plt.xlabel('실제 통행량 구간')
plt.ylabel('중력모델 CPC')
plt.xticks(rotation=45)

plt.subplot(2, 2, 3)
plt.scatter(
    테스트총량['실제내부통행'],
    테스트총량['예측내부통행_최종OD대각'],
    alpha=0.6,
    label='내부통행 (최종 OD 대각)',
    marker='o',
)
plt.scatter(
    테스트총량['실제내부통행'],
    테스트총량['예측내부통행_LGBM'],
    alpha=0.6,
    label='내부통행 (LGBM)',
    marker='x',
)
최댓값 = max(
    테스트총량['실제내부통행'].max(),
    테스트총량['예측내부통행_최종OD대각'].max(),
    테스트총량['예측내부통행_LGBM'].max(),
)
plt.plot([0, 최댓값], [0, 최댓값], 'r--')
plt.title('내부통행: LGBM 예측값 vs 최종 OD 대각')
plt.xlabel('실제 내부통행')
plt.ylabel('예측 내부통행')
plt.legend()

plt.subplot(2, 2, 4)
plt.scatter(
    테스트OD['실제통행량'] + 1,
    테스트OD['예측통행량_중력모델'] + 1,
    alpha=0.05,
    color='green',
    s=2,
)
plt.plot([1, 100000], [1, 100000], 'r--')
plt.xscale('log')
plt.yscale('log')
plt.title('실제 vs 예측 OD (중력모델, 로그 스케일)')
plt.xlabel('실제 통행량 (+1)')
plt.ylabel('예측 통행량 (+1)')

plt.tight_layout()
그림경로 = 출력폴더 / '중력모델_성능분석.png'
plt.savefig(그림경로, dpi=300)
plt.show()

print('그림 저장:', 그림경로)